# 01 · Data audit — from snapshot to analysis sets

**Goal:** check scope, units, missingness and duplicate candidates before interpreting rents. Preserve `data/processed/listings.csv`. Do not apply table-wide `dropna()` or impute during EDA.

Review decisions are stored in `data/analysis/review_queue.csv`. Initial decisions were made by Codex using titles and CSV fields; original webpages were not verified. Edit `scope_decision`, `price_decision` and `area_decision` using `keep/exclude/unresolved`, add `review_note` and `reviewer`, then **Run All**. Do not edit the context columns used to validate the snapshot. Original listing text and place names stay in their source language.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if not (ROOT / "data/processed/listings.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "data/processed/listings.csv").exists():
    raise FileNotFoundError("Open the notebook from the project root or notebooks/")
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)


## 1. Input, schema and recorded dates

In [2]:
from src.eda_audit import run_audit, audit_summary, markdown_table
from src.eda_reporting import missing_summary
raw = pd.read_csv(ROOT / "data/processed/listings.csv")
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing_n": raw.isna().sum()}))
display(pd.crosstab(raw.source, raw.category))
print("Input:", raw.shape)
print("Posting dates:", raw.posted_date.min(), "→", raw.posted_date.max())
print("Recorded parser dates:", raw.scraped_at.unique(), "— download dates are not verified")

,dtype,missing_n
source,str,0
listing_id,str,0
category,str,0
url,str,0
title,str,0
price_vnd,int64,0
price_raw,str,0
area_m2,float64,399
area_raw,float64,399
bedrooms,float64,973


category,can_ho_chung_cu,nha_rieng,nha_tro
source,,,
alonhadat,231,224,209
nhatot,317,189,800


Input: (1970, 18)
Posting dates: 2026-09-07 → 2026-09-19
Recorded parser dates: <StringArray>
['2026-09-19']
Length: 1, dtype: str — download dates are not verified


## 2. Missingness by source and property type

In [3]:
display(missing_summary(raw))

area_m2  bedrooms  toilets  furniture
source    category                                              
alonhadat can_ho_chung_cu      0.0       0.0     65.4       68.8
          nha_rieng            0.0      30.4     91.5       79.5
          nha_tro              0.0      37.3     97.6       43.5
nhatot    can_ho_chung_cu      4.4       2.8     28.1       37.5
          nha_rieng            5.3       9.5     37.0       76.2
          nha_tro             46.9     100.0    100.0       13.2

## 3. Apply the audit and review decisions

- Raw price text must agree with numeric values and monthly units. Four per-m² listings already contain total rents in the input; do not multiply again.
- Keywords only flag records. Reviewed or unresolved office/whole-building/shared-bed listings are separated from the main whole-unit/room set.
- Unclear area/bedroom/bathroom values become missing in `*_analysis` fields; missing features alone do not remove the entire row.
- P1/P99 are review flags, not automatic exclusions. Private houses are excluded from the primary price/m² set.
- Duplicate groups are candidates based on ID/URL or normalized title + location + category, not verified unique properties.

In [4]:
analysis = run_audit(ROOT)
display(analysis.price_unit_status.value_counts().rename_axis("status").to_frame("n"))
display(analysis.quality_flags.str.split(";").explode().loc[lambda x: x.ne("")].value_counts().to_frame("n"))
display(analysis.groupby(["scope_flag", "review_status"], dropna=False).size().to_frame("n"))

,n
status,
monthly_verified,1960
review_excluded,6
per_m2_total_verified,4


,n
quality_flags,
area_missing_or_invalid,399
scope_suspicious,146
price_tail_p01_p99,39
bedrooms_unit_ambiguous,37
duplicate_candidate,28
shared_or_per_person,20
price_needs_review,9
area_needs_review,4
category_title_conflict,3


n
scope_flag review_status      
normal     not_required   1814
           pending           3
           reviewed          7
suspicious pending          55
           reviewed         91

## 4. Pending reviews and repost candidates

In [5]:
queue = pd.read_csv(ROOT / "data/analysis/review_queue.csv").fillna("")
display(queue.loc[queue.review_status.eq("pending"), ["listing_id", "title", "price_raw", "quality_flags", "review_note"]])
display(queue.loc[queue.duplicate_group_id.ne(""), ["listing_id", "title", "duplicate_group_id"]])

,listing_id,title,price_raw,quality_flags,review_note
16,nt_134723389,"TÂY SƠN – LÔ GÓC 35M² 4 TẦNG – THÔNG SÀN – KINH DOANH – Ô TÔ – 13,5TR","13,5 triệu/tháng",scope_suspicious,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
22,nt_134713716,"Nhà nguyên căn mặt phố Bùi Ngọc Dương, Hai Bà Trưng, kinh doanh","15,5 triệu/tháng",scope_suspicious,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
25,nt_134701849,"Cho thuê nhà phố Đào Tấn, Ba Đình, kinh doanh nhỏ",24 triệu/tháng,scope_suspicious,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
26,nt_134125006,"CHung cư MINI 5 sao ĐH Ngoại Thương, Luật 5tr - Tủ lạnh, giường Tầng",5 triệu/tháng,scope_suspicious;shared_or_per_person,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
33,nt_134714619,"Phòng trọ Đống Đa, 1 Ngủ, Full nội thất, Ô tô đỗ cửa",500.000 đ/tháng,area_missing_or_invalid;price_needs_review;price_tail_p01_p99,The price is unusually low or its unit is unclear; verify the original listing before using it a...
34,nt_134714540,"Phòng trọ giường tầng gần Chợ Thành Công, Láng Hạ - 5tr2","5,2 triệu/tháng",scope_suspicious;shared_or_per_person;area_missing_or_invalid,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
46,nt_133942526,nhà 3 tầng 4 phòng ngủ,14 triệu/tháng,scope_suspicious;category_title_conflict;price_tail_p01_p99,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
56,nt_132767940,CHO THUÊ PHÒNG Ở TẠI TRƯỜNG CHINH GẦN NGÃ TƯ SỞ VÀ TÔN THẤT TÙNG,150.000 đ/tháng,price_needs_review;price_tail_p01_p99,The price is unusually low or its unit is unclear; verify the original listing before using it a...
59,nt_134566520,Phòng Studio Giường Tầng Full Đồ Đống Đa 4tr9,"4,9 triệu/tháng",scope_suspicious;shared_or_per_person;area_missing_or_invalid,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."
63,alo_18988315,"TẬP THỂ TẦNG 1 THÀNH CÔNG, BA ĐÌNH - MẶT NGÕ KD - 30M RA Ô TÔ – HÀNG ĂN, KHO, PHÒNG TẬP",Giá: 8 triệu / tháng,scope_suspicious,"CSV context is insufficient to establish residential/commercial use, whole-room/shared-place ren..."


,listing_id,title,duplicate_group_id
1,nt_129301773,Cho Thuê Căn Hộ Dịch Vụ Studio Cao Cấp 35m² Âu Cơ Tây Hồ Hà Nội,candidate_nt_128039288
2,nt_134714512,"Cho thuê căn hộ mặt đường Đào Duy Anh, Đống Đa, full nội thất",candidate_nt_134714510
3,nt_134714510,"Cho thuê căn hộ mặt đường Đào Duy Anh, Đống Đa, full nội thất",candidate_nt_134714510
8,nt_132487403,"Cho thuê CHDV Studio mới tinh Ngõ 690 Lạc Long Quân, gần Lotte Tây Hồ",candidate_nt_132487359
10,nt_132487359,"Cho thuê CHDV Studio mới tinh Ngõ 690 Lạc Long Quân, gần Lotte Tây Hồ",candidate_nt_132487359
14,nt_128039288,Cho Thuê Căn Hộ Dịch Vụ Studio Cao Cấp 35m² Âu Cơ Tây Hồ Hà Nội,candidate_nt_128039288
38,nt_134713875,Phòng trọ full đồ Hoàng Mai,candidate_nt_134713557
39,nt_134713841,"Phòng trọ mới đủ đồ ban công Phan Trọng Tuệ, Đại Thanh",candidate_nt_134713839
40,nt_134713839,"Phòng trọ mới đủ đồ ban công Phan Trọng Tuệ, Đại Thanh",candidate_nt_134713839
42,nt_134713557,Phòng trọ full đồ Hoàng Mai,candidate_nt_134713557


## 5. Finalize the analysis sets

`include_price`: valid monthly asking rent and eligible scope under the audit policy. `include_ppm2`: the apartment/room subset with usable area. Each question requires only its relevant variables; there is no universal complete-case dataset.

Keep excluded rows and their reasons in `listings_analysis.csv`. Notebook 02 only reads this dataset; return to the audit if a new issue is found.

In [6]:
assert len(analysis) == len(raw)
pd.testing.assert_frame_equal(raw, analysis[raw.columns], check_dtype=True)
assert (analysis.include_ppm2 <= analysis.include_price).all()
assert analysis.loc[analysis.include_price, "price_month_vnd"].gt(0).all()
assert not analysis.loc[analysis.include_ppm2, "category"].eq("nha_rieng").any()
display(audit_summary(analysis))
display(analysis.groupby(["source", "category"]).agg(input_n=("listing_id", "size"), price_n=("include_price", "sum"), ppm2_n=("include_ppm2", "sum")))
print("Exported the analysis dataset, review queue and reports/data_dictionary.md")

,stage,n
0,Input,1970
1,Flagged (including missing area),614
2,Review pending,58
3,Excluded from price set,124
4,Final residential price set,1846
5,Final primary price/m² set,1140


input_n  price_n  ppm2_n
source    category                                 
alonhadat can_ho_chung_cu      231      227     227
          nha_rieng            224      139       0
          nha_tro              209      206     206
nhatot    can_ho_chung_cu      317      314     300
          nha_rieng            189      183       0
          nha_tro              800      777     407

Exported the analysis dataset, review queue and reports/data_dictionary.md
